In [ ]:
import json
import csv
from pyproj import Transformer

CITY_FILE  = 'map-data/aa_building_footprints.json'
UM_FILE    = 'map-data/um-building-footprint-edited.geojson'
CSV_FILE   = 'map-data/resource_new.csv'

In [ ]:
with open(CITY_FILE) as f:
    city_data = json.load(f)

features = city_data['features']
print(f'Total city features: {len(features)}')
print(f'Top-level keys: {list(city_data.keys())}')
print(f'Geometry type: {city_data["geometryType"]}')
print(f'Spatial reference: {city_data["spatialReference"]}')
print()
print('Attribute columns:', list(features[0]['attributes'].keys()))

In [ ]:
TARGET_FIDS = {'BLD-034725', 'BLD-004706'}

found = {}
for feat in features:
    fid = feat['attributes'].get('FacilityID', '')
    if fid in TARGET_FIDS:
        found[fid] = feat
        print(json.dumps(feat['attributes'], indent=2))
        print(f'  Ring count: {len(feat["geometry"]["rings"])}')
        print(f'  Points in ring 0: {len(feat["geometry"]["rings"][0])}')
        print()

In [ ]:
transformer = Transformer.from_crs('EPSG:2253', 'EPSG:4326', always_xy=True)

def reproject_rings(rings):
    """Convert Esri ring coordinates from EPSG:2253 (ft) to WGS84 [lon, lat]."""
    result = []
    for ring in rings:
        result.append([list(transformer.transform(x, y)) for x, y in ring])
    return result

def ring_center(rings_wgs84):
    coords = rings_wgs84[0]
    return (
        sum(c[1] for c in coords) / len(coords),  # lat
        sum(c[0] for c in coords) / len(coords),  # lon
    )

for fid, feat in found.items():
    rings = reproject_rings(feat['geometry']['rings'])
    lat, lon = ring_center(rings)
    print(f"{feat['attributes']['Bldg_Name']} ({fid})")
    print(f'  Center: {lat:.6f}, {lon:.6f}')
    print(f'  Google Maps: https://maps.google.com/?q={lat:.6f},{lon:.6f}')
    print()

In [ ]:
# Metadata for each city building — building_id reuses the FacilityID from the city dataset
BUILDING_META = {
    'BLD-034725': {
        'building_id':   'BLD-034725',
        'building_name': 'Maynard Parking Structure',
        'resource_name': 'Center for Academic Innovation',
    },
    'BLD-004706': {
        'building_id':   'BLD-004706',
        'building_name': 'Plymouth Park',
        'resource_name': "Michigan Alzheimer's Disease Center",
    },
}

with open(UM_FILE) as f:
    um_data = json.load(f)

print(f'UM features before: {len(um_data["features"])}')

for fid, meta in BUILDING_META.items():
    feat = found[fid]
    rings_wgs84 = reproject_rings(feat['geometry']['rings'])

    new_feature = {
        'type': 'Feature',
        'properties': {
            'building_id':   meta['building_id'],
            'building_name': meta['building_name'],
            'is_resource':   True,
        },
        'geometry': {
            'type': 'Polygon',
            'coordinates': rings_wgs84,
        }
    }
    um_data['features'].append(new_feature)
    print(f"Added: {meta['building_name']} (building_id={meta['building_id']}) → {meta['resource_name']}")

print(f'\nUM features after: {len(um_data["features"])}')

In [ ]:
with open(UM_FILE, 'w') as f:
    json.dump(um_data, f)

print(f'Saved: {UM_FILE}')

In [ ]:
ID_UPDATES = {
    'Center for Academic Innovation':       'BLD-034725',
    "Michigan Alzheimer's Disease Center":  'BLD-004706',
}

with open(CSV_FILE, newline='') as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = list(reader)

for row in rows:
    if row['resource_name'] in ID_UPDATES:
        old_id = row['building_id']
        row['building_id'] = ID_UPDATES[row['resource_name']]
        print(f"{row['resource_name']}: '{old_id}' → '{row['building_id']}'")

with open(CSV_FILE, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print('\nCSV updated.')

## 7. Verify

In [ ]:
# Confirm new features exist in geojson
with open(UM_FILE) as f:
    um_check = json.load(f)

new_ids = {'BLD-034725', 'BLD-004706'}
added = [f['properties'] for f in um_check['features'] if f['properties'].get('building_id') in new_ids]
print('New GeoJSON features:')
for p in added:
    print(f"  {p['building_name']} | id={p['building_id']} | is_resource={p['is_resource']}")

# Confirm CSV building_ids updated
print()
with open(CSV_FILE, newline='') as f:
    rows = list(csv.DictReader(f))
targets = [r for r in rows if r['resource_name'] in ID_UPDATES]
print('Updated CSV rows:')
for r in targets:
    print(f"  {r['resource_name']} | building_id={r['building_id']}")